In [7]:
# Install required libraries
!pip install langchain
!pip install langchain-community
!pip install langchain-google-genai
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install faiss-cpu
!pip install pypdf





   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 9.9 MB/s eta 0:00:00


In [9]:
# Gemini model for LangChain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# PDF loader
from langchain_community.document_loaders import PyPDFLoader

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector database
from langchain_community.vectorstores import FAISS

import os




In [20]:
# Add your Gemini API key here
os.environ["GOOGLE_API_KEY"] = "GEMINI_API_KEY"


In [11]:
from google.colab import files

# Upload your PDF file
uploaded = files.upload()

# Get uploaded file name
pdf_file = list(uploaded.keys())[0]


Saving class 11 chemistry.pdf to class 11 chemistry.pdf


In [12]:
# Load the PDF
loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Total pages loaded:", len(documents))


Total pages loaded: 133


In [13]:
# Split large text into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print("Total chunks created:", len(docs))


Total chunks created: 303


In [14]:
# Convert text chunks into embeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)


In [ ]:
# Store embeddings in FAISS vector database

vector_db = FAISS.from_documents(docs, embeddings)

print("Vector database created successfully")


In [39]:
# Load Gemini model

model = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=0.3
)


In [ ]:
while True:

    query = input("Ask a question from the lab manual: ")

    # Find relevant document chunks
    relevant_docs = vector_db.similarity_search(query)

    # Combine retrieved text
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # Create prompt
    prompt = f"""
    Answer the question based only on the following lab manual content.

    Content:
    {context}

    Question:
    {query}
    """

    # Generate answer using Gemini
    response = model.invoke(prompt)

    print("\nAnswer:", response.content)

